# 🚀 Phase 1 — Part 5: Train Whisper (Fine-Tuning with Checkpoint Saving)

---

## ⭐ THIS IS THE MOST IMPORTANT NOTEBOOK
This notebook fine-tunes OpenAI's Whisper on your Bangla dialect data. It is the longest-running notebook and the one most likely to be interrupted by Colab disconnects.

## 🔒 Checkpoint Strategy (How Nothing Gets Lost)

| What | How It's Saved | Where |
|------|---------------|-------|
| Model weights after each epoch | `save_strategy='epoch'` | `models/whisper-small-{dialect}/checkpoint-{step}/` |
| Best model (lowest WER) | `load_best_model_at_end=True` | `models/whisper-small-{dialect}/` (final save) |
| Processor/tokenizer | `processor.save_pretrained()` | Same folder as model |
| Training state (optimizer, scheduler) | Saved with each checkpoint | Inside checkpoint folder |

### If Colab disconnects mid-training:
1. Re-open this notebook
2. Run all cells from top
3. The training cell uses `resume_from_checkpoint=True` → it finds the latest checkpoint in Drive and continues
4. **You lose ZERO progress** — training picks up from the last saved epoch

### What is inside each checkpoint folder?
```
checkpoint-500/
├── config.json            ← Model architecture config
├── model.safetensors      ← Model weights (~1-2 GB)
├── optimizer.pt           ← Optimizer state (for resume)
├── scheduler.pt           ← Learning rate scheduler state
├── trainer_state.json     ← Epoch, step, best metric
├── training_args.bin      ← All training hyperparameters
└── rng_state.pth          ← Random state (exact reproducibility)
```

---

## Step 0: Mount Drive, Load Config, Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('❌ NO GPU! Go to Runtime → Change runtime type → T4 GPU')

## Step 1: Install All Dependencies
These must be installed every time you start a new Colab session.

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate
print('✅ All packages installed')

## Step 2: Choose Model Size and Dialect

### Whisper sizes comparison:
| Size | Parameters | VRAM Needed | Training Time (1hr data) | Accuracy |
|------|-----------|-------------|--------------------------|----------|
| tiny | 39M | ~2 GB | ~20 min | Lowest |
| base | 74M | ~3 GB | ~40 min | Low |
| small | 244M | ~5 GB | ~2 hours | Good |
| medium | 769M | ~11 GB | ~6 hours | Best |

**Recommended:** Start with `small` — best balance of accuracy and speed for T4 GPU.

Run this cell **once per model/dialect combination.** For example:
1. First run: `MODEL_SIZE = 'small'`, `DIALECT = 'dhaka'`
2. Second run: Change to `DIALECT = 'chittagong'` and run all training cells again

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these two values for each training run
# ════════════════════════════════════════════════════════════════

MODEL_SIZE = 'small'     # Options: 'tiny', 'base', 'small', 'medium'
DIALECT    = 'dhaka'     # Options: 'dhaka', 'chittagong'

# ── Derived paths (auto-calculated) ──
MODEL_ID = f'openai/whisper-{MODEL_SIZE}'
OUT_DIR  = os.path.join(BASE, f'models/whisper-{MODEL_SIZE}-{DIALECT}')

print(f'🎯 Model:   {MODEL_ID}')
print(f'🗣️ Dialect: {DIALECT}')
print(f'💾 Output:  {OUT_DIR}')

# Check if there's a previous checkpoint (for resume)
if os.path.exists(OUT_DIR):
    checkpoints = [d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        print(f'\n🔄 Found {len(checkpoints)} existing checkpoints! Training will RESUME.')
        print(f'   Latest: {sorted(checkpoints)[-1]}')
    else:
        print(f'\n📁 Output folder exists but no checkpoints found. Starting fresh.')
else:
    print(f'\n🆕 No previous training found. Starting from scratch.')

## Step 3: Load Whisper Processor and Model

### What are these?
- **Processor**: Converts raw audio → model input features (mel spectrogram) and converts model output IDs → text
- **Model**: The neural network that does the actual speech-to-text conversion
- `forced_decoder_ids`: Forces the model to generate Bangla text (language code 'bn')
- `suppress_tokens = []`: Don't suppress any tokens during generation

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print(f'🔄 Loading {MODEL_ID}...')
processor = WhisperProcessor.from_pretrained(
    MODEL_ID, language='bn', task='transcribe'
)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Force Bangla output
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='bn', task='transcribe'
)
model.config.suppress_tokens = []

print(f'✅ Model loaded: {MODEL_ID}')
print(f'   Parameters: {model.num_parameters() / 1e6:.0f}M')

## Step 4: Load and Prepare Dataset

### What happens here:
1. Load the train and validation CSVs from Drive
2. Convert the `file_path` column to an Audio feature (HuggingFace auto-loads WAVs)
3. Apply the Whisper processor to convert audio → input features and text → label IDs

### The `prepare_batch` function:
- Reads the audio array from the WAV file
- Converts it to a mel spectrogram (what Whisper expects as input)
- Tokenizes the transcript text into token IDs (what Whisper learns to predict)

In [ ]:
from datasets import Dataset, Audio
import pandas as pd

# Load CSVs
train_csv = os.path.join(BASE, f'dataset/{DIALECT}_train.csv')
val_csv   = os.path.join(BASE, f'dataset/{DIALECT}_val.csv')

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
print(f'📊 Train: {len(train_df)} samples')
print(f'📊 Val:   {len(val_df)} samples')

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

# Cast file_path to Audio (auto-loads WAV files at 16kHz)
train_ds = train_ds.cast_column('file_path', Audio(sampling_rate=16000))
val_ds   = val_ds.cast_column('file_path', Audio(sampling_rate=16000))

print('✅ Dataset loaded and audio column configured')

In [ ]:
# ════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION — Convert audio + text to model inputs
# ════════════════════════════════════════════════════════════════

def prepare_batch(batch):
    """Convert raw audio → mel spectrogram features and text → token IDs."""
    audio = batch['file_path']['array']
    
    # Audio → input features (mel spectrogram)
    batch['input_features'] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]
    
    # Text → token IDs (labels)
    batch['labels'] = processor.tokenizer(
        batch['transcript']
    ).input_ids
    
    return batch

print('🔄 Processing training data...')
train_ds = train_ds.map(
    prepare_batch,
    remove_columns=train_ds.column_names,
    num_proc=1  # Use 1 for Colab stability
)

print('🔄 Processing validation data...')
val_ds = val_ds.map(
    prepare_batch,
    remove_columns=val_ds.column_names,
    num_proc=1
)

print(f'✅ Features extracted!')
print(f'   Train samples: {len(train_ds)}')
print(f'   Val samples:   {len(val_ds)}')

## Step 5: Data Collator

### What is a Data Collator?
When the trainer picks a batch of samples (e.g., 8 samples), they might have different lengths. The collator:
1. **Pads** input features to the same length
2. **Pads** labels to the same length
3. Replaces padding tokens in labels with `-100` (so they are ignored in loss calculation)

In [ ]:
import dataclasses
from typing import Any, Dict, List, Union

@dataclasses.dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], Any]]]) -> Dict[str, Any]:
        # Pad input features
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )
        
        # Pad labels
        label_features = [f['labels'] for f in features]
        labels_batch = self.processor.tokenizer.pad(
            {'input_ids': label_features}, return_tensors='pt'
        )
        
        # Replace padding with -100 (ignored by loss function)
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch['input_ids'] == self.processor.tokenizer.pad_token_id, -100
        )
        batch['labels'] = labels
        return batch

collator = WhisperDataCollator(processor=processor)
print('✅ Data collator ready')

## Step 6: WER Metric

### What is WER (Word Error Rate)?
- Measures how different the predicted text is from the correct text
- **Lower is better** — 0% = perfect, 100% = completely wrong
- WER = (Substitutions + Insertions + Deletions) / Total Words in Reference
- This is computed on the validation set after every epoch

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    """Calculate WER on validation set predictions."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # Replace -100 with pad token for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    # Decode IDs → text
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 4)}

print('✅ WER metric loaded')

## Step 7: Training Arguments

### 🔒 KEY CHECKPOINT SETTINGS EXPLAINED:

| Setting | Value | What It Does |
|---------|-------|-------------|
| `output_dir` | Google Drive path | Checkpoints saved to Drive (persistent!) |
| `save_strategy='epoch'` | After every epoch | Saves full checkpoint every epoch |
| `save_total_limit=3` | Keep last 3 | Older checkpoints deleted to save Drive space |
| `load_best_model_at_end=True` | End of training | Loads the best checkpoint (lowest WER) |
| `metric_for_best_model='wer'` | WER | Best = lowest Word Error Rate |
| `gradient_checkpointing=True` | Saves GPU memory | Trades compute for memory — essential for T4 |
| `fp16=True` | Half precision | Faster training, less memory |

### Batch size math:
- `per_device_train_batch_size=8` × `gradient_accumulation_steps=2` = **effective batch size of 16**
- If you get OOM (out of memory), reduce batch size to 4 and increase accumulation to 4

In [ ]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,                        # 💾 SAVE TO GOOGLE DRIVE!
    
    # ── Training hyperparameters ──
    per_device_train_batch_size=8,              # Samples per GPU per step
    gradient_accumulation_steps=2,              # Effective batch = 8 × 2 = 16
    learning_rate=1e-5,                         # Learning rate
    warmup_steps=100,                           # Gradual LR warmup
    num_train_epochs=15,                        # Total epochs
    
    # ── Memory optimization ──
    gradient_checkpointing=True,                # ESSENTIAL for T4 GPU
    fp16=torch.cuda.is_available(),             # Half precision (faster)
    
    # ── 🔒 CHECKPOINT SAVING (most important part!) ──
    evaluation_strategy='epoch',                # Evaluate after every epoch
    save_strategy='epoch',                      # Save checkpoint every epoch
    save_total_limit=3,                         # Keep only last 3 checkpoints
    load_best_model_at_end=True,                # Auto-load best model when done
    metric_for_best_model='wer',                # Best = lowest WER
    greater_is_better=False,                    # Lower WER is better
    
    # ── Generation settings ──
    predict_with_generate=True,                 # Required for Seq2Seq evaluation
    generation_max_length=225,                  # Max output tokens
    
    # ── Logging ──
    logging_steps=25,                           # Log every 25 steps
    logging_dir=os.path.join(OUT_DIR, 'logs'),  # TensorBoard logs
    report_to='none',                           # Set to 'wandb' if you have wandb
    run_name=f'whisper-{MODEL_SIZE}-{DIALECT}',
)

print('✅ Training arguments configured')
print(f'   Effective batch size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}')
print(f'   Epochs: {args.num_train_epochs}')
print(f'   Checkpoints saved to: {OUT_DIR}')
print(f'   Keeping last {args.save_total_limit} checkpoints')

## Step 8: Create Trainer and START Training

### 🔄 RESUME LOGIC:
- `resume_from_checkpoint=True` tells the Trainer to look for existing checkpoints in `OUT_DIR`
- If found → training resumes from the latest checkpoint (epoch, optimizer state, everything)
- If not found → training starts from scratch
- You don't need to change anything — it's automatic!

### ⏱️ Expected training time:
- ~360 clips × 15 epochs ≈ **1.5–3 hours** on T4 GPU for Whisper-small
- Progress is printed every 25 steps
- WER is shown after each epoch

### 🚨 If you get `CUDA out of memory`:
1. Change `per_device_train_batch_size` to **4** in Step 7
2. Change `gradient_accumulation_steps` to **4**
3. Re-run Steps 7 and 8

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

# ════════════════════════════════════════════════════════════════
# 🚀 START TRAINING (with automatic resume!)
# ════════════════════════════════════════════════════════════════

# Check for existing checkpoints
resume_ckpt = None
if os.path.exists(OUT_DIR):
    checkpoints = [d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
        resume_ckpt = os.path.join(OUT_DIR, latest)
        print(f'🔄 RESUMING from: {resume_ckpt}')
    else:
        print('🆕 No checkpoints found. Starting fresh training.')
else:
    print('🆕 Starting fresh training.')

print(f'\n{'='*60}')
print(f'  Training whisper-{MODEL_SIZE} on {DIALECT} dialect')
print(f'  Checkpoints saving to: {OUT_DIR}')
print(f'{'='*60}\n')

trainer.train(resume_from_checkpoint=resume_ckpt)

print('\n✅ Training complete!')

## Step 9: Save Final Model & Processor to Google Drive

### What gets saved here:
- **Model weights** (`model.safetensors` or `pytorch_model.bin`) — the trained neural network
- **Processor files** (`tokenizer.json`, `preprocessor_config.json`, etc.) — needed to use the model
- Without the processor, you **cannot** use the model for inference later

### Why save separately?
The checkpoints contain optimizer state too (which is large). This final save contains **only what's needed for inference** — smaller and cleaner.

In [ ]:
# ════════════════════════════════════════════════════════════════
# SAVE FINAL MODEL + PROCESSOR TO GOOGLE DRIVE
# ════════════════════════════════════════════════════════════════

# Save model (this is the BEST model — automatically loaded at end of training)
trainer.save_model(OUT_DIR)
print(f'✅ Model saved to: {OUT_DIR}')

# Save processor (tokenizer + feature extractor)
processor.save_pretrained(OUT_DIR)
print(f'✅ Processor saved to: {OUT_DIR}')

# List what was saved
print(f'\n📁 Contents of {OUT_DIR}/')
for item in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, item)
    if os.path.isdir(full):
        print(f'  📁 {item}/')
    else:
        size_mb = os.path.getsize(full) / (1024*1024)
        print(f'  📄 {item} ({size_mb:.1f} MB)')

## Step 10: Quick Test — Does the Model Work?

Let's test the fine-tuned model on one clip to make sure it's working.

In [ ]:
from transformers import pipeline as hf_pipeline

# Load the saved model from Drive
print(f'🔄 Loading fine-tuned model from: {OUT_DIR}')
test_asr = hf_pipeline(
    'automatic-speech-recognition',
    model=OUT_DIR,
    device=0
)

# Pick a test clip
test_csv = os.path.join(BASE, f'dataset/{DIALECT}_test.csv')
if os.path.exists(test_csv):
    test_df = pd.read_csv(test_csv)
    if len(test_df) > 0:
        sample = test_df.iloc[0]
        result = test_asr(sample['file_path'])
        print(f'\n🎯 Test Result:')
        print(f'   Audio:      {os.path.basename(sample["file_path"])}')
        print(f'   Expected:   {sample["transcript"]}')
        print(f'   Predicted:  {result["text"]}')
    else:
        print('Test CSV is empty')
else:
    print('Test CSV not found')

## Step 11: Save Training Script to Drive

In [ ]:
# Save a reference copy of the training script
script_path = os.path.join(BASE, 'scripts/train_whisper.py')
with open(script_path, 'w') as f:
    f.write(f'''# train_whisper.py — Fine-tune Whisper on Bangla dialect data
# Generated by Phase1_Part5 notebook
# Usage: Change MODEL_SIZE and DIALECT, then run all cells

MODEL_SIZE = '{MODEL_SIZE}'  # tiny | base | small | medium
DIALECT = '{DIALECT}'        # dhaka | chittagong
MODEL_ID = f'openai/whisper-{{MODEL_SIZE}}'
OUT_DIR = f'models/whisper-{{MODEL_SIZE}}-{{DIALECT}}'

# See Phase1_Part5_Train_Whisper.ipynb for full code
''')
print(f'✅ Script reference saved to: {script_path}')

---
## ✅ Part 5 Complete!

### What is now saved in Google Drive:
| File | Location | What |
|------|----------|------|
| Fine-tuned model weights | `models/whisper-{size}-{dialect}/model.safetensors` | The trained model |
| Processor/tokenizer | `models/whisper-{size}-{dialect}/` | Required to use the model |
| Training checkpoints | `models/whisper-{size}-{dialect}/checkpoint-*/` | For resume (auto-cleaned) |
| Training logs | `models/whisper-{size}-{dialect}/logs/` | Loss/WER history |
| Script reference | `scripts/train_whisper.py` | Code backup |

### How to load this model again (in any future notebook):
```python
from transformers import pipeline
asr = pipeline('automatic-speech-recognition',
               model='/content/drive/MyDrive/NSU_499A/models/whisper-small-dhaka',
               device=0)
result = asr('path_to_audio.wav')
print(result['text'])
```

### Next steps:
1. **Change `DIALECT = 'chittagong'`** in Step 2 and re-run ALL cells to train on Chittagong
2. Then move to **Part 6** (Train Wav2Vec2) or **Part 7** (Train WavLM)

---